# 第13讲 · 无监督学习与 k-means 聚类

本 notebook 对应课件 `lecture13.md` 与脚本 `kmeans_clustering.py`。

**一句话目标**：给定一堆没有标签的二维点，自动把它们分成几团（聚类）。

完整流程：
1. 造二维分团数据
2. `KMeans` 聚类（标签 / 中心 / 惯性）
3. 预测新点
4. 四步手算对照
5. 原始中心点影响（k-means++ vs 随机）
6. 肘部法则选 K
7. 聚类散点图
8. numpy 手写简化 k-means 对比

依赖：`numpy`、`scikit-learn`、`matplotlib`。

In [ ]:
# 让绘图直接嵌入 notebook（不弹窗）
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from plots import setup_zh_font

from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

# 中文字体设置
setup_zh_font()

## 0. 造数据 + 标准化

`make_blobs` 生成 150 个围绕 3 个中心（`centers=3`）的二维点，`cluster_std=1.2` 控制离散程度。

**标准化** `StandardScaler().fit_transform`：把每维均值归 0、方差归 1，让量纲一致（课件常见错误表强调过）。

In [ ]:
X, _ = make_blobs(n_samples=150, centers=3, cluster_std=1.2,
                  random_state=42, center_box=(-10, 10))
X = StandardScaler().fit_transform(X[:, :2])
print(f"数据 shape={X.shape}")

## 1. 最小 k-means：课件代码演示

`KMeans(n_clusters=3, n_init=10, random_state=42)` 训练后：
- `labels_`：每个点属于哪个簇
- `cluster_centers_`：簇中心坐标
- `inertia_`：惯性（簇内点到中心的平方距离之和，越小越紧凑）
- `predict`：预测新点属于哪个簇

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=42)
km.fit(X)
print("每个点属于哪个簇 (前10个):", km.labels_[:10])
print("簇中心点:\n", np.round(km.cluster_centers_, 3))
print(f"惯性 inertia_: {km.inertia_:.3f}")

new = np.array([-1.0, 0.5])
print(f"新点 {new} 属于簇 {km.predict(new[:, np.newaxis].T)[0]}")

## 2. 四步手算对照：中心点会"漂移"到簇内均值

k-means 迭代的本质就四步：**选初始中心 → 分配（每个点归最近中心）→ 更新（中心=簇内均值）→ 重复**。用课件的 10 点小数据，`k=2` 能直观看到中心如何收敛。

In [ ]:
mini = np.array([[1, 1], [1.5, 2], [2, 1.5], [2.5, 2], [2, 3],
                 [8, 8], [8.5, 8.5], [9, 8], [8.5, 9], [8, 9.5]])
km2 = KMeans(n_clusters=2, n_init=10, random_state=42).fit(mini)
print("[手算对照] 课件 10 点数据，k=2:")
print("  簇标签:", km2.labels_)
print("  中心点（已漂移到簇内均值）:\n", np.round(km2.cluster_centers_, 3))

## 3. 重点：原始中心点的影响（随机初始 vs k-means++）

k-means 对**初始中心点敏感**，不同初始可能收敛到不同局部最优。
- `k-means++`：sklearn 默认，选择相距较远的初始中心，更稳、更易收敛到优解。
- `random`：随机初始化，可能陷入较差的局部最优（惯性更大）。

> 同一数据两次跑结果不同 → 多半是初始中心随机所致。稳定方法：设 `random_state`、加大 `n_init` 或默认 k-means++。

In [ ]:
km_pp = KMeans(n_clusters=3, init="k-means++", n_init=1,
               random_state=42).fit(X)
km_rand = KMeans(n_clusters=3, init="random", n_init=1,
                 random_state=7).fit(X)
print("[中心点影响]")
print(f"  k-means++  惯性 = {km_pp.inertia_:8.3f} (更稳、更易收敛到优解)")
print(f"  随机初始   惯性 = {km_rand.inertia_:8.3f}")

## 4. 肘部法则选 K

惯性随 K 增大而单调减小，但降低速度有**拐点**（肘）。拐点附近 K 最合理——K 再大收益变小，反而增加复杂度。

In [ ]:
print("[肘部法则] K -inertia 曲线（找拐点）")
Ks = range(1, 8)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).inertia_
            for k in Ks]
for k, v in zip(Ks, inertias):
    print(f"  K={k}  惯性={v:.3f}")
print("  惯性随 K 增大而减小；由陡变缓的『拐点』处 K 最合理。")

### 肘部法则可视化

把 K-惯性曲线画出来，能更清楚地看到拐点位置。

In [ ]:
plt.figure(figsize=(6.4, 4.2))
plt.plot(list(Ks), inertias, "o-", color="#3a7ca5")
plt.xlabel("K（簇数）")
plt.ylabel("惯性 inertia")
plt.title("肘部法则：K - inertia 曲线")
plt.xticks(list(Ks))
plt.grid(alpha=0.4)
plt.show()

## 5. 用 numpy 手写简化 k-means 对比

不调库实现 k-means 核心循环：
1. 随机挑 k 个样本作初始中心
2. 分配：每个点归到最近中心（广播算距离）
3. 更新：每簇中心 = 簇内点均值
4. 中心不再变化则终止

与 `sklearn` 对比惯性，验证手写正确性。

In [ ]:
def kmeans_numpy(X, k, n_iter=100, random_state=42):
    """返回 (labels, centers, inertia)。不依赖 sklearn.kmeans。"""
    rng = np.random.default_rng(random_state)
    centers = X[rng.choice(len(X), k, replace=False)]   # 1. 随机初始中心
    labels = np.zeros(len(X), dtype=int)
    for _ in range(n_iter):
        # 2. 分配：每个点归到最近中心（广播算距离平方）
        d2 = np.sum((X[:, None, :] - centers[None, :, :]) ** 2, axis=2)
        new_labels = d2.argmin(axis=1)
        # 3. 更新：每簇中心 = 簇内点均值
        new_centers = centers.copy()
        for c in range(k):
            mask = new_labels == c
            if mask.any():
                new_centers[c] = X[mask].mean(axis=0)
        # 4. 迭代终止：中心不再变化
        if np.allclose(new_centers, centers) and np.array_equal(new_labels, labels):
            break
        centers, labels = new_centers, new_labels
    inertia = float(np.sum(np.min(np.sum((X[:, None, :] - centers[None, :, :]) ** 2, axis=2), axis=1)))
    return labels, centers, inertia

In [ ]:
labels_hand, centers_hand, inert_hand = kmeans_numpy(X, k=3, random_state=42)
labels_sk, centers_sk, inert_sk = km.labels_, km.cluster_centers_, km.inertia_
print("[numpy 手写 k-means vs sklearn]")
print(f"  手写 惯性 = {inert_hand:8.3f}    sklearn 惯性 = {inert_sk:8.3f}")

## 6. 聚类散点图

按簇着色的散点图 + 红色叉号标出簇中心，直观看出聚类效果。

In [ ]:
plt.figure(figsize=(6.4, 5.2))
plt.scatter(X[:, 0], X[:, 1], c=km.labels_, s=40, cmap="viridis", alpha=0.7)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c="red", marker="x", s=200, linewidths=3, label="簇中心")
plt.title("k-means 聚类 (K=3) 按簇着色 + 中心点")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

## 小结

- k-means 是无监督聚类，核心是「选中心 → 分配 → 更新」的迭代。
- 惯性越小，簇越紧凑；肘部法则帮我们选合理的 K。
- 初始中心影响结果，k-means++ 或固定随机种子能提高稳定性。
- 手写版验证了算法本质。